##ЛР6. Кластеризация: k-means, DBSCAN, иерархические методы

[10 баллов]

## О формате сдачи (все ЛР)

🔷 **<font color='plum'>При решении используйте данный шаблон</font>**

    ✅ Можно добавлять новые ячейки любых типов
    ❌ Не нужно удалять текстовые ячейки c разметкой частей ноутбука и формулировками заданий


🔷 **<font color='plum'>При оценивании задач учитывается код</font>**

    ✅ Задания, в которых необходим код, могут помечаться фразами "Your code here"/"Ваш код" и аналогичными
    ❌ Ответы на вопросы без сопутствующего кода оцениваются в 0 баллов
    ❌ Наличие работоспособного кода в ноутбуке, если на сказано иного, обязательно

🔷 **<font color='plum'>При оценивании задач учитываются выводы</font>**

    ✅ Задания, в которых необходимы выводы, могут помечаться фразами "Вывод"/"Ответ на вопрос"/"Ваш текст" и аналогичными
    ✅ Выводы подразумевают текстовый (письменный, не устный) ответ (можно писать markdown, latex).
    ✅ Сопутствующие изображения, графики, таблички - приветствуются!
    ❌ При отсутствии выводов задание не засчитается на полный балл

🔷 **<font color='plum'>Устная защита</font>**

    ❌❌❌ Если не пройдена устная защита по ЛР, то баллы не начисляются вовсе
    (вопросы устной защиты охватывают лекционный материал и практический материал по заданиям ЛР, список вопросов не публикуется)

## Этапы

<font color='DarkOrange'>**Задание [баллов: 10]:**</font>

Поставьте задачу кластеризации, используя датасет из ЛР начала семестра (датасет с Kaggle или другого портала): сформируем её от классификации. Выберите категориальный признак, который показывает классы объектов, пусть будет от 3 до 6 классов, оставьте только объекты, которые относятся к ним. Далее оставляем только числовые признаки (часть тоже можете удалить, пусть далее используются 5-10 числовых  признаков).



Деления train\test не будет - решаем задачу без учителя.



Попробуйте получить кластеры, которые соответствуют исходным классам. Задайте функционал качества, по которому будете определять эффективность (сравнивать состав кластеров и классов).


Методы: k-means (собственная реализация и из sklearn), любой вариант иерархической кластеризации (из любой библиотеки), DBSCAN (из любой библиотеки).


Попробуйте улучшить результат, используя нормализацию (выберите подходящий вариант сами) и/или уменьшение размерности (метод PCA - метод главных компонент) как предобработку с одним из методов кластеризации.


Начисление баллов за использование методов:
- [важно!]выбор признака, классы по которому будут в работе (1),
- k-means собственной реализации (3),
- k-means из библиотеки (1),
- иерархический метод из библиотеки (1),
- DBSCAN из библиотеки (1),
- предобработка уменьшением размерности (PCA) из библиотеки (1),
- предобработка нормализацией (1),
- выводы по итогам (1)



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    v_measure_score,
    silhouette_score,
    davies_bouldin_score
)
from scipy.optimize import linear_sum_assignment
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 6)

In [2]:
df = pd.read_csv("women_in_stem.csv")
display(df.head())
print(df.info())

,Country,Year,Female Enrollment (%),Female Graduation Rate (%),STEM Fields,Gender Gap Index
0,China,2018,20.4,43.2,Engineering,0.52
1,China,2005,35.6,29.3,Mathematics,0.98
2,China,2005,53.7,32.4,Biology,0.60
3,Germany,2007,65.0,63.6,Mathematics,0.69
4,Canada,2010,54.4,28.8,Engineering,0.74


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Country                     500 non-null    object 
 1   Year                        500 non-null    int64  
 2   Female Enrollment (%)       500 non-null    float64
 3   Female Graduation Rate (%)  500 non-null    float64
 4   STEM Fields                 500 non-null    object 
 5   Gender Gap Index            500 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 23.6+ KB
None


In [3]:
df_encoded = df.copy()

country_encoder = LabelEncoder()
df_encoded["Country_encoded"] = country_encoder.fit_transform(df_encoded["Country"])

country_mapping = dict(
    zip(country_encoder.classes_, country_encoder.transform(country_encoder.classes_))
)

print("\nCountry -> Country_encoded:")
for country, code in country_mapping.items():
    print(f"{country}: {code}")

df_encoded = df_encoded.drop(columns=["Country"])


Соответствие Country -> Country_encoded:
Australia: 0
Canada: 1
China: 2
Germany: 3
India: 4
USA: 5


In [ ]:
# Истинные классы (для сравнения с кластерами)
y_true = df["STEM Fields"].copy()

print("Распределение классов STEM Fields:")
print(y_true.value_counts())
print("Кол-во классов:", y_true.nunique())

In [ ]:
# Матрица признаков: только числовые признаки (5 штук)
feature_cols = ["Year", "Female Enrollment (%)", "Female Graduation Rate (%)", "Gender Gap Index", "Country_encoded"]
X = df_encoded[feature_cols].copy()

print("X shape:", X.shape)
display(X.head())

In [ ]:
def evaluate_clustering(X, y_true, labels, name="model"):
    # External (сравнение с классами)
    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)
    vms = v_measure_score(y_true, labels)

    # Internal (структура кластеров по X) — считаем аккуратно
    uniq = np.unique(labels)
    # Silhouette определён, только если >=2 кластеров и нет случая "все в один кластер"
    sil = None
    dbi = None
    try:
        if len(uniq) >= 2 and not (len(uniq) == 1):
            sil = silhouette_score(X, labels)
            dbi = davies_bouldin_score(X, labels)
    except Exception:
        pass

    result = {
        "model": name,
        "ARI": ari,
        "NMI": nmi,
        "V-measure": vms,
        "Silhouette": sil,
        "Davies-Bouldin": dbi
    }
    return result